# Phase 4 - Variational Quantum Classifier: Angle Encoding

A **variational quantum classifier (VQC)** has three parts:

1. A **feature map** that encodes the data into a quantum state (this is the
   part that changes between notebooks - here it is **angle encoding**).
2. A trainable **ansatz**, a circuit with adjustable rotation parameters.
3. A classical **optimizer** that tunes those parameters to reduce
   classification error.

Angle encoding uses one rotation per feature and has **no entangling gates**, so it is the natural non-entangling counterpart to the ZZ feature map in the hypothesis.

We reuse the exact dataset, split, and scaling from Phase 2 so the result is
directly comparable to the classical baseline and to the other encodings.

In [ ]:
"""04_vqc_angle_encoding.ipynb"""

# Cell 01 - Imports and reproducible seed

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit.circuit.library import real_amplitudes, z_feature_map
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils import algorithm_globals

import qml_utils as hu

algorithm_globals.random_seed = hu.RANDOM_SEED
np.random.seed(hu.RANDOM_SEED)

---
**Cell 02 - Load and scale the data (same as Phase 2).**

In [ ]:
# Cell 02 - Prepare the two-moons data, scaled to [0, pi]

x_train, x_test, y_train, y_test = hu.load_moons(n_samples=200, noise=0.20)
x_train_scaled, x_test_scaled, scaler = hu.scale_features(
    x_train, x_test, feature_range=(0.0, np.pi)
)
print(f"Train: {x_train_scaled.shape}, Test: {x_test_scaled.shape}")

---
**Cell 03 - Build the encoding and the ansatz.**
The feature map is `z_feature_map` (angle encoding, no entanglement). The
trainable part is `real_amplitudes`, a standard hardware-efficient ansatz of
$R_y$ rotations and CX gates. We draw the combined circuit and record its
depth for the Phase 7 comparison.

In [ ]:
# Cell 03 - Feature map + ansatz

feature_map = z_feature_map(feature_dimension=2, reps=2)
ansatz = real_amplitudes(num_qubits=2, reps=2)

full_circuit = feature_map.compose(ansatz)
print(f"Number of qubits        : {full_circuit.num_qubits}")
print(f"Trainable parameters    : {ansatz.num_parameters}")
print(f"Circuit depth (composed): {full_circuit.decompose().depth()}")
display(full_circuit.draw(output="mpl"))

---
**Cell 04 - Train the classifier.**
We use **COBYLA**, a gradient-free optimizer that works well for small VQCs.
A callback records the objective value at each step so we can watch training
converge. Training runs entirely on the local simulator and should take only
a few seconds.

In [ ]:
# Cell 04 - Train the VQC and record the objective at each step

objective_values = []


def store_objective(weights, value):
    objective_values.append(value)


vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=80),
    callback=store_objective,
)

vqc.fit(x_train_scaled, y_train)

angle_train_acc = vqc.score(x_train_scaled, y_train)
angle_test_acc = vqc.score(x_test_scaled, y_test)
print(f"Angle-encoding VQC training accuracy: {angle_train_acc:.3f}")
print(f"Angle-encoding VQC test accuracy    : {angle_test_acc:.3f}")

---
**Cell 05 - Training convergence.**
The objective value should trend downward and then flatten. A noisy or
non-decreasing curve is itself a finding: it signals that this encoding and
ansatz are hard to optimize, which is part of what we report.

In [ ]:
# Cell 05 - Plot the optimization curve

plt.figure(figsize=(7, 4))
plt.plot(objective_values)
plt.xlabel("optimizer iteration")
plt.ylabel("objective value")
plt.title("Angle-encoding VQC training convergence")
plt.grid(True, alpha=0.3)
plt.show()

---
**Cell 06 - Decision boundary.**
Compare this boundary to the classical SVM from Phase 2. Does angle encoding
produce a curved boundary that follows the moons, or something simpler?

In [ ]:
# Cell 06 - Plot the learned decision boundary (grid kept small for speed)

hu.plot_decision_boundary(
    vqc.predict,
    x_test_scaled,
    y_test,
    title=f"Angle-encoding VQC (test accuracy {angle_test_acc:.2f})",
    grid_steps=40,
)
plt.show()

---
## Interpretation

Record the **test accuracy** and note the shape of the decision boundary.
Because angle encoding has no entangling gates, it is our non-entangling
reference point. In Phase 7 we will place it next to amplitude encoding, the
ZZ feature map, and the classical SVM.